In [ ]:
# ANALYSIS GRU
# SEARCH WORDS :
# YEARLY_GRU_ESTIMATION
# GRU_FITTED_TARGET_SPACE_PERFORMANCE
# GRU_SCORE_CALIBRATION
# GRU_THRESHOLD_CALIBRATION
# GRU_THRESHOLD_TRADER_CALIBRATION

import pandas as pd
import numpy as np
import warnings
import os, sys
import pickle

from pathlib import Path
from arch.utility.exceptions import ConvergenceWarning


warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=ConvergenceWarning)


sys.path.insert(0,"/Users/shah/CODE_BOOK_4/THESIS_WORKING/THESIS_2")
sys.path.insert(0,"/Users/shah/CODE_BOOK_4/code_document/REPOSITORIES/MFIV_calibration")

os.chdir("/Users/shah/CODE_BOOK_4/THESIS_WORKING/THESIS_2")

sys.modules.pop("srcB", None)


# =============================================================================================================
# ======================================= LOAD BASE OUTPUTS ===================================================
# =============================================================================================================

with open("/Users/shah/CODE_BOOK_4/code_document/REPOSITORIES/data/IS/outputs_by_year/outputs_by_year_04.pkl","rb") as f:
    outputs_by_year = pickle.load(f)


# =============================================================================================================
# ======================================= IMPORTS FROM SRC ====================================================
# =============================================================================================================

from srcB.market import *
from srcB.helpers import *


# =============================================================================================================
# ======================================= READ D_BTC AND DF ===================================================
# =============================================================================================================

DATA_DIR = Path("../data/rawData")

YEARS = [2019, 2020, 2021]
QUARTERS = ["q1", "q2", "q3", "q4"]

d_btc_raw_by_year = {}
df_raw_by_year = {}

for y in YEARS:
    d_btc_raw_by_year[y] = pd.read_parquet(DATA_DIR/f"btc_underlying_minutely_{y}.parquet")
    df_raw_by_year[y] = pd.read_parquet(DATA_DIR/f"derivative_data_{y}.parquet")


# =============================================================================================================
# ======================================= SET INDEX TO DATETIME ===============================================
# =============================================================================================================

for y in YEARS:
    d_btc_raw_by_year[y] = d_btc_raw_by_year[y].set_index(pd.to_datetime(d_btc_raw_by_year[y]["date_time"]))
    df_raw_by_year[y] = df_raw_by_year[y].set_index(pd.to_datetime(df_raw_by_year[y]["date_time"]))


# =============================================================================================================
# ======================================= LOAD X_BY_YEAR ======================================================
# =============================================================================================================

with open("/Users/shah/CODE_BOOK_4/code_document/REPOSITORIES/data/IS/x_by_year_fut/x_by_year.pkl", "rb") as f:
    X_by_year = pd.read_pickle(f)

for y in YEARS:
    X_by_year[y] = X_by_year[y].copy()
    X_by_year[y]["rv"] = X_by_year[y]["rt"] ** 2

Xq_by_year = {}

for y in YEARS:
    Xq_by_year[y] = {}
    Xq_by_year[y]["q1"] = X_by_year[y].loc[f"{y}-01-01" : f"{y}-03-31 23:59:59"]
    Xq_by_year[y]["q2"] = X_by_year[y].loc[f"{y}-04-01" : f"{y}-06-30 23:59:59"]
    Xq_by_year[y]["q3"] = X_by_year[y].loc[f"{y}-07-01" : f"{y}-09-30 23:59:59"]
    Xq_by_year[y]["q4"] = X_by_year[y].loc[f"{y}-10-01" : f"{y}-12-31 23:59:59"]


# =============================================================================================================
# ======================================= MFIV SCALING ========================================================
# =============================================================================================================
# CONVERTS ANNUALIZED MFIV -> CUMULATIVE H-HOUR DECIMAL VARIANCE

MFIVS = list(range(2, 577))

for y in YEARS:
    for q in QUARTERS:
        for H in MFIVS:
            Xq_by_year[y][q][f"MFIV_{H}"] *= (H / 8760.0)

for y in YEARS:
    for q in QUARTERS:
        Xq_by_year[y][q] = Xq_by_year[y][q].loc[:, ~Xq_by_year[y][q].columns.duplicated(keep="first")]


# =============================================================================================================
# ======================================= BORROW RFR COLUMN ===================================================
# =============================================================================================================

with open("/Users/shah/CODE_BOOK_4/code_document/REPOSITORIES/data/IS/ready_data/d_btc_by_year_all.pkl", "rb") as f:
    d_btc_by_year = pickle.load(f)

for y in YEARS:
    d_btc_raw_by_year[y]["rfr"] = d_btc_by_year[y]["rfr"].reindex(d_btc_raw_by_year[y].index, method="ffill")


# =============================================================================================================
# ======================================= CHECKPOINT 1 ========================================================
# =============================================================================================================

In [ ]:
# =============================================================================================================
# ======================================= GRU PARAMETERS ======================================================
# =============================================================================================================

with open("/Users/shah/CODE_BOOK_4/code_document/REPOSITORIES/data/IS/hyperparameters/gru_best_params.pkl", "rb") as f:
    best_gru_params = pickle.load(f)

VAL_FRACTION = 0.20
EPOCHS = 40
PATIENCE = 6
SEED = 11

In [ ]:
# =============================================================================================================
# ======================================= YEARLY_GRU_ESTIMATION ===============================================
# =============================================================================================================

sys.modules.pop("srcB.gru_model", None)
from srcB.gru_model import *

gru_bundle_by_year = {}
gru_targets_by_year = {}
gru_output_by_year = {}

for y in YEARS:
    X_df = pd.concat([Xq_by_year[y][q] for q in QUARTERS]).sort_index().copy()
    garch_df = pd.concat([outputs_by_year[y][q]["garch"] for q in QUARTERS]).sort_index()
    X_df["garch"] = garch_df.reindex(X_df.index)

    gru_output, gru_bundle, gru_targets = gru_mfiv(
        X_df=X_df,
        option_df=df_raw_by_year[y],
        time_df=d_btc_raw_by_year[y],
        sequence_length=168,
        forward_hours=24,
        delta_target=0.20,
        val_fraction=VAL_FRACTION,
        hidden_size=best_gru_params["hidden_size"],
        num_layers=best_gru_params["num_layers"],
        dropout=best_gru_params["dropout"],
        epochs=EPOCHS,
        batch_size=best_gru_params["batch_size"],
        learning_rate=best_gru_params["learning_rate"],
        weight_decay=best_gru_params["weight_decay"],
        patience=PATIENCE,
        seed=SEED
    )
    gru_output_by_year[y] = gru_output
    gru_bundle_by_year[y] = gru_bundle
    gru_targets_by_year[y] = gru_targets

    for q in QUARTERS:
        q_index = Xq_by_year[y][q].index
        gru_q = gru_output.reindex(q_index)
        outputs_by_year[y][q] = outputs_by_year[y][q].drop(columns=gru_q.columns, errors="ignore")
        outputs_by_year[y][q] = pd.concat([outputs_by_year[y][q], gru_q], axis=1)

In [ ]:
# =============================================================================================================
# ================================ GRU_FITTED_TARGET_SPACE_PERFORMANCE ========================================
# =============================================================================================================

target_cols = [
    "target_long_straddle",
    "target_short_straddle",
    "target_long_strangle",
    "target_short_strangle"
]

rows = []

for y in YEARS:
    test = pd.concat([
        gru_output_by_year[y][["gru_action"]],
        gru_targets_by_year[y][target_cols]
    ], axis=1, sort=False).dropna()

    realized = pd.Series(0.0, index=test.index)

    for action in ["long_straddle", "short_straddle", "long_strangle", "short_strangle"]:
        mask = test["gru_action"] == action
        realized.loc[mask] = test.loc[mask, f"target_{action}"]

    sharpe = realized.mean() / realized.std()

    rows.append({
        "year": y,
        "n": len(realized),
        "mean_target": realized.mean(),
        "target_sharpe": sharpe,
        "profitable": (realized > 0).mean()
    })

target_sharpe = pd.DataFrame(rows)
print(target_sharpe)

   year     n  mean_target  target_sharpe  profitable
0  2019  2194    -0.000046      -0.074655    0.000000
1  2020  4093     0.002359       0.092324    0.290252
2  2021  5008     0.004237       0.171305    0.550120


In [ ]:
# =============================================================================================================
# ======================================= GRU_SCORE_CALIBRATION ===============================================
# =============================================================================================================

rows = []

for y in YEARS:
    test = pd.concat([
        gru_output_by_year[y][["gru_action", "gru_best_score"]],
        gru_targets_by_year[y][target_cols]
    ], axis=1, sort=False).dropna()

    test = test[test["gru_action"] != "no_trade"].copy()
    test["target_pnl"] = 0.0

    for action in ["long_straddle", "short_straddle", "long_strangle", "short_strangle"]:
        m = test["gru_action"] == action
        test.loc[m, "target_pnl"] = test.loc[m, f"target_{action}"]

    test["score_bin"] = pd.qcut(
        test["gru_best_score"],
        5,
        labels=False,
        duplicates="drop"
    )

    x = (
        test.groupby("score_bin")
        .agg(
            n=("target_pnl", "size"),
            mean_score=("gru_best_score", "mean"),
            mean_target=("target_pnl", "mean"),
            sharpe=("target_pnl", lambda z: z.mean() / z.std()),
            profitable=("target_pnl", lambda z: (z > 0).mean())
        )
        .reset_index()
    )

    x["year"] = y
    rows.append(x)

score_check = pd.concat(rows, ignore_index=True)

print(score_check[[
    "year", "score_bin", "n", "mean_score",
    "mean_target", "sharpe", "profitable"
]].to_string(index=False))

 year  score_bin   n  mean_score  mean_target    sharpe  profitable
 2019          0   4    0.000022    -0.011382 -4.642532    0.000000
 2019          1   3    0.000055    -0.004476 -1.532203    0.000000
 2019          2   4    0.000092    -0.003672 -1.425083    0.000000
 2019          3   3    0.000111    -0.004539 -1.436958    0.000000
 2019          4   4    0.000121    -0.003515 -1.207470    0.000000
 2020          0 478    0.000322     0.003622  0.152450    0.573222
 2020          1 478    0.000894     0.001839  0.068668    0.552301
 2020          2 477    0.001434     0.002936  0.145102    0.561845
 2020          3 478    0.002089     0.002383  0.111422    0.447699
 2020          4 478    0.002981     0.009424  0.162100    0.351464
 2021          0 890    0.000756     0.000608  0.035517    0.585393
 2021          1 890    0.002307     0.001431  0.065908    0.611236
 2021          2 889    0.004516     0.001404  0.055645    0.584927
 2021          3 890    0.007635     0.003330  0

In [ ]:
# =============================================================================================================
# ======================================= GRU_THRESHOLD_CALIBRATION ===========================================
# =============================================================================================================

tests = {}

for y in YEARS:
    x = pd.concat([
        gru_output_by_year[y][["gru_action", "gru_best_score"]],
        gru_targets_by_year[y][target_cols]
    ], axis=1, sort=False).dropna()

    x = x[x["gru_action"] != "no_trade"].copy()
    x["target_pnl"] = 0.0

    for action in ["long_straddle", "short_straddle", "long_strangle", "short_strangle"]:
        m = x["gru_action"] == action
        x.loc[m, "target_pnl"] = x.loc[m, f"target_{action}"]

    tests[y] = x

scores_all = pd.concat([tests[y]["gru_best_score"] for y in YEARS])

rows = []

for q in [0, .20, .40, .50, .60, .70, .80, .85, .90]:

    threshold = scores_all.quantile(q)

    for y in YEARS:
        x = tests[y][tests[y]["gru_best_score"] >= threshold]
        sharpe = x["target_pnl"].mean() / x["target_pnl"].std()

        rows.append({
            "quantile": q,
            "threshold": threshold,
            "year": y,
            "n": len(x),
            "mean_target": x["target_pnl"].mean(),
            "sharpe": sharpe
        })

threshold_check = pd.DataFrame(rows)

print(threshold_check.to_string(index=False))

 quantile    threshold  year    n  mean_target    sharpe
     0.00 1.216831e-07  2019   18    -0.005629 -1.403331
     0.00 1.216831e-07  2020 2389     0.004041  0.121204
     0.00 1.216831e-07  2021 4449     0.004770  0.182082
     0.20 9.759676e-04  2019    0          NaN       NaN
     0.20 9.759676e-04  2020 1610     0.004178  0.113872
     0.20 9.759676e-04  2021 3875     0.005460  0.201525
     0.40 1.917676e-03  2019    0          NaN       NaN
     0.40 1.917676e-03  2020  834     0.006002  0.128962
     0.40 1.917676e-03  2021 3280     0.006304  0.220262
     0.50 2.494496e-03  2019    0          NaN       NaN
     0.50 2.494496e-03  2020  433     0.011700  0.196281
     0.50 2.494496e-03  2021 2995     0.006804  0.234396
     0.60 3.369193e-03  2019    0          NaN       NaN
     0.60 3.369193e-03  2020   91     0.038453  0.361541
     0.60 3.369193e-03  2021 2652     0.007286  0.246083
     0.70 4.938703e-03  2019    0          NaN       NaN
     0.70 4.938703e-03  2020   

In [ ]:
# the pooled quantile grid is not suitable yet for GRU threshold selection.
# 2019’s entire positive-score distribution sits below the pooled 20th-percentile threshold 0.000905, so every candidate above zero eliminates 2019 completely. That is very different from the TCN case.
# The key issue to determine is whether there are sensible common absolute GRU thresholds that still allow 2019 observations.
# =============================================================================================================
# ======================================= GRU SCORE DISTRIBUTION ==============================================
# =============================================================================================================

for y in YEARS:
    x = tests[y]["gru_best_score"]

    print(
        y,
        "n:", len(x),
        "min:", x.min(),
        "q25:", x.quantile(.25),
        "q50:", x.quantile(.50),
        "q75:", x.quantile(.75),
        "q90:", x.quantile(.90),
        "max:", x.max()
    )

2019 n: 18 min: 3.538793e-06 q25: 4.501896910369396e-05 q50: 9.271467570215464e-05 q75: 0.00011410168372094631 q90: 0.00011968535836786032 max: 0.00012448942
2020 n: 2389 min: 3.3993274e-07 q25: 0.0007392074912786484 q50: 0.0014219093136489391 q75: 0.0022882558405399323 q90: 0.00288831195794046 max: 0.0046791965
2021 n: 4449 min: 1.2168312e-07 q25: 0.0018299502553418279 q50: 0.004470466170459986 q75: 0.008546348661184311 q90: 0.013827764987945558 max: 0.031197991


In [ ]:
os.makedirs("/Users/shah/CODE_BOOK_4/code_document/REPOSITORIES/data/IS/gru", exist_ok=True)
with open("/Users/shah/CODE_BOOK_4/code_document/REPOSITORIES/data/IS/gru/gru_bundle_by_year.pkl", "wb") as f:
    pickle.dump(gru_bundle_by_year, f)

print(gru_bundle_by_year.keys())

dict_keys([2019, 2020, 2021])


In [ ]:
# =============================================================================================================
# ======================================= PREPARE TRADER INPUT ================================================
# =============================================================================================================

for y in YEARS:
    for q in QUARTERS:
        outputs_by_year[y][q]["rv"] = Xq_by_year[y][q]["rv"].reindex(
            outputs_by_year[y][q].index
        )
        outputs_by_year[y][q]["MFIV_24"] = Xq_by_year[y][q]["MFIV_24"].reindex(
            outputs_by_year[y][q].index
        )

In [ ]:
# =============================================================================================================
# ======================================= GRU TRADER ADAPTER ==================================================
# =============================================================================================================
# GRU signals are not altered;
# it only copies them into column names the existing trader expects.
for y in YEARS:
    for q in QUARTERS:
        outputs_by_year[y][q]["tcn_action"] = outputs_by_year[y][q]["gru_action"]
        outputs_by_year[y][q]["tcn_best_score"] = outputs_by_year[y][q]["gru_best_score"]

In [ ]:
# =============================================================================================================
# ======================================= CACHE FOR WARMUP ====================================================
# =============================================================================================================

CACHE_YEARS = [2019, 2020, 2021]

trader_cache = {
    y: pd.concat(
        [
            outputs_by_year[yy][q]
            for yy in CACHE_YEARS
            if yy <= y
            for q in QUARTERS
        ]
    ).sort_index()
    for y in CACHE_YEARS
}

In [ ]:
# =============================================================================================================
# ======================================= GRU_THRESHOLD_TRADER_CALIBRATION ===================================
# =============================================================================================================

sys.modules.pop("srcB.trader_testing_tcn", None)
from srcB.trader_testing_tcn import testing_execute_trading_strategy

THRESHOLDS = [0.0, 0.0001, 0.0002, 0.0003, 0.0005]

rows = []

for threshold in THRESHOLDS:
    for y in YEARS:

        market = Market(
            option_df=df_raw_by_year[y],
            time_df=d_btc_raw_by_year[y],
            starting_cash=100_000,
            im_multiplier=0.15,
            mm_buffer_rate=0.05,
        )

        snapshots, greek_tests = testing_execute_trading_strategy(
            bt_year=y,
            bt_quarter=None,
            model="tcn_mfiv",
            outputs_by_year=outputs_by_year,
            market=market,
            trader_cache=trader_cache,
            run_by="year",
            tcn_score_threshold=threshold
        )

        equity = pd.Series(
            [s["equity"] for s in snapshots],
            index=[s["time"] for s in snapshots]
        )

        returns = equity.pct_change().dropna()
        sharpe = returns.mean() / returns.std() * np.sqrt(24 * 365)

        rows.append({
            "threshold": threshold,
            "year": y,
            "trades": len(greek_tests),
            "sharpe": sharpe
        })

threshold_trader_check = pd.DataFrame(rows)

print(threshold_trader_check.to_string(index=False))

 threshold  year  trades   sharpe
    0.0000  2019       6 0.449999
    0.0000  2020     191 0.856358
    0.0000  2021     273 0.949240
    0.0001  2019       2 0.220370
    0.0001  2020     200 0.675760
    0.0001  2021     269 0.986724
    0.0002  2019       0      NaN
    0.0002  2020     194 0.734126
    0.0002  2021     267 0.994926
    0.0003  2019       0      NaN
    0.0003  2020     183 0.738099
    0.0003  2021     268 0.948825
    0.0005  2019       0      NaN
    0.0005  2020     177 0.713162
    0.0005  2021     265 0.725755
